Find missing numbers in a given dataframe sequence of numbers

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import max, col

data_inp = [1,2,3,4,5,10]
            
df_inp = spark.createDataFrame(data_inp, schema=["inp_num"])

display(df_inp)

# display(df_inp)
df_min = df_inp.agg({"inp_num": "min"}).collect()[0]['min(inp_num)']
df_max = df_inp.agg(max(col("inp_num"))).collect()[0]['max(inp_num)']
# display(df_max)

data = [] 
for i in range(df_min,df_max + 1):
  # data.append(Row(i))
  data.append(i)

df = spark.createDataFrame(data, schema=["series_num"])
# display(df)

df_missing = df.join(df_inp, df.series_num == df_inp.inp_num, how="left") \
            .filter(df_inp.inp_num.isNull())  \
            .select(df.series_num)

display(df_missing)

Cricket Match Summary 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, count, sum, when

data = [
    ('India', 'SL', 'India'),
    ('SL', 'Aus', 'Aus'),
    ('SA', 'Eng', 'Eng'),
    ('Eng', 'NZ', 'NZ'),
    ('Aus', 'India', 'India')
]

schema = StructType([
    StructField('Team_1', StringType(), True),
    StructField('Team_2', StringType(), True),
    StructField('Winner', StringType(), True)
])

icc_world_cup_df = spark.createDataFrame(data, schema)
display(icc_world_cup_df)

df_team1 = icc_world_cup_df.select((col('Team_1')).alias('Team'),col('Winner'))
df_team2 = icc_world_cup_df.select((col('Team_2')).alias('Team'),col('Winner'))
df_union = df_team1.union(df_team2)

df_pre = df_union.withColumn(
    'is_win',
    when(col('Winner') == col('Team'), 1).otherwise(0)
).groupBy(col('Team')).agg(
    count('*').alias('Total_no_of_matches_played'),
    sum('is_win').alias('Total_no_of_wins')
)

df_final = df_pre.withColumn(
    'no_of_losses', 
    col('Total_no_of_matches_played') - col('Total_no_of_wins')) \
    .orderBy(col('Total_no_of_wins').desc())

display(df_final)

SQL Way - To solve above Pyspark problem - Cricket Summary

In [0]:
%sql
create table icc_world_cup
(
Team_1 Varchar(20),
Team_2 Varchar(20),
Winner Varchar(20)
);
INSERT INTO icc_world_cup values('India','SL','India');
INSERT INTO icc_world_cup values('SL','Aus','Aus');
INSERT INTO icc_world_cup values('SA','Eng','Eng');
INSERT INTO icc_world_cup values('Eng','NZ','NZ');
INSERT INTO icc_world_cup values('Aus','India','India');

select * 
from icc_world_cup;

WITh CTE AS (
select team_1 as team, winner
from icc_world_cup
union All
select team_2 as team, winner
from icc_world_cup)
SELECT * from cte;

SELECT team, 
  count(*) total_number_matches,
  SUM(CASE WHEN team = Winner then 1 else 0 END) as number_of_wins,
  count(*) - SUM(CASE WHEN team = Winner then 1 else 0 END) as number_of_loss
from cte
group by team

Display Job Summary

In [0]:
data = [
    ("Job1", "table1", 100),
    ("Job1", "table1", 50),
    ("Job2", "table1", 20),
    ("Job3", "table1", 60),
    ("Job10", "table2", 50),
    ("Job20", "table2", 60),
    ("Job30", "table2", 15),
    ("Job40", "table2", 5)
]

columns = ["Input", "Name", "running time(mins)"]

df = spark.createDataFrame(data, columns)
display(df)

from pyspark.sql import Window
from pyspark.sql.functions import col, max, min, first

window_max = Window.partitionBy("Name")
window_min = Window.partitionBy("Name")
# df_max = df.withColumn("max_time", max("running time(mins)").over(window_max))
# df_min = df.withColumn("min_time", min("running time(mins)").over(window_min))

df_max = df.withColumn("max_time", max("running time(mins)").over(window_max)) \
    .filter(col("running time(mins)") == col("max_time")) \
    .groupBy("Name") \
    .agg(first("Input").alias("Job_Max_Load_time"))

df_min = df.withColumn("min_time", min("running time(mins)").over(window_min)) \
    .filter(col("running time(mins)") == col("min_time")) \
    .groupBy("Name") \
    .agg(first("Input").alias("Job_Min_Load_time"))

result = df_max.join(df_min, "Name").select(
    col("Name").alias("table Name"),
    col("Job_Max_Load_time").alias("Job - Max Load time"),
    col("Job_Min_Load_time").alias("Job- Min Load time")
)
display(result)

Find top 2 revenue genrrating products in each category

In [0]:
from pyspark.sql import Row

data = [
    Row(product='A', category='Mobile', qty=3, price=200),
    Row(product='B', category='Mobile', qty=1, price=700),
    Row(product='C', category='Mobile', qty=2, price=500),
    Row(product='D', category='Laptop', qty=1, price=2000),
    Row(product='E', category='Laptop', qty=4, price=1500),
    Row(product='F', category='Laptop', qty=1, price=1000)
]

df = spark.createDataFrame(data)
df_revenue = df.withColumn("revenue", col("price") * col("qty"))
display(df_revenue)

from pyspark.sql import Window
from pyspark.sql.functions import col, row_number

window_shop = Window.partitionBy("category").orderBy(col("revenue").desc())
df_rank = df_revenue.withColumn("rank", row_number().over(window_shop)).filter(col("rank") <= 2)
display(df_rank)


New and Repeat customer by day

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType
from datetime import datetime

data = [
    (1, 100, datetime.strptime('2022-01-01', '%Y-%m-%d').date(), 2000),
    (2, 200, datetime.strptime('2022-01-01', '%Y-%m-%d').date(), 2500),
    (3, 300, datetime.strptime('2022-01-01', '%Y-%m-%d').date(), 2100),
    (4, 100, datetime.strptime('2022-01-02', '%Y-%m-%d').date(), 2000),
    (5, 400, datetime.strptime('2022-01-02', '%Y-%m-%d').date(), 2200),
    (6, 500, datetime.strptime('2022-01-02', '%Y-%m-%d').date(), 2700),
    (7, 100, datetime.strptime('2022-01-03', '%Y-%m-%d').date(), 3000),
    (8, 400, datetime.strptime('2022-01-03', '%Y-%m-%d').date(), 1000),
    (9, 600, datetime.strptime('2022-01-03', '%Y-%m-%d').date(), 3000)
]

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("order_amount", IntegerType(), True)
])

df = spark.createDataFrame(data, schema=schema)
display(df)

from pyspark.sql.functions import col, sum, when, min

# We know which customer ordered for the first time...
df_first_order = df.groupBy("customer_id").agg(min(col("order_date")).alias("first_order_date"))

df_new_old_cust = df.join(df_first_order, on=["customer_id"], how="inner") \
.withColumn("repeat_customer_flag", when(col("order_date") > col("first_order_date"), 1).otherwise(0)) \
.withColumn("new_customer_flag", when(col("order_date") == col("first_order_date"), 1).otherwise(0)) \
.groupBy(col("order_date")) \
.agg(sum(col("new_customer_flag")).alias("new_customers"), sum(col("repeat_customer_flag")).alias("repeat_customers")) \
.orderBy(col("order_date"))


display(df_new_old_cust)

SQL Way to find new and repeat customers

In [0]:
%sql
create table customer_orders (
order_id integer,
customer_id integer,
order_date date,
order_amount integer
);

insert into customer_orders values
(1,100,cast('2022-01-01' as date),2000),
(2,200,cast('2022-01-01' as date),2500),
(3,300,cast('2022-01-01' as date),2100),
(4,100,cast('2022-01-02' as date),2000),
(5,400,cast('2022-01-02' as date),2200),
(6,500,cast('2022-01-02' as date),2700),
(7,100,cast('2022-01-03' as date),3000),
(8,400,cast('2022-01-03' as date),1000),
(9,600,cast('2022-01-03' as date),3000)
;

select * from customer_orders;

with first_cte AS (
select customer_id, min(order_date) as first_order_date
from customer_orders
group by customer_id
)
SELECT co.order_date,
SUM(Case when order_date = first_order_date then 1 else 0 end) as new_customer_flag,
SUM(CASE WHEN order_date > first_order_date then 1 else 0 end) as repeated_flag
from customer_orders co
join first_cte fc
on co.customer_id = fc.customer_id
group by co.order_date